# Embedding 모델별 PaCMAP 군집 리포트

영화 plot summary 임베딩 결과(`result_data_2/result-{model}.csv`)를 온톨로지 라벨(`origin_data/movie_ontology.csv`)과 결합해,
모델별 **genre / theme(category) / mood** 군집 분리도를 2D·3D PaCMAP으로 시각화한 결과입니다.

> PaCMAP 입력은 cosine 거리와 맞추기 위해 L2 정규화한 임베딩을 사용합니다.

## 자료구조

| 파일 | 컬럼 |
|------|------|
| `origin_data/movie_ontology.csv` | `movie_id`, `title`, `genres[]`, `themes[]`, `moods[]` |
| `result_data_2/result-{model}.csv` | `movie_id`, `title`, `summary`, `genres`, `embedding_0` … `embedding_{d-1}` |

> `genres`는 result CSV에서, `themes`·`moods`는 `movie_id`로 ontology와 결합합니다.

## 모델별 속도와 메모리 사용량
| 모델 | 메모리 사용량 | 속도 |
|------|------|------|
| BAAI/bge-m3 | 2.5 GB | 25item/s |
| Qwen/Qwen3-Embedding-0.6B | 1.5 GB | 10item/s |
| Qwen/Qwen3-Embedding-4B | 8 GB | 2.5item/s |


In [1]:
# 필요 시 의존성 설치 (최초 1회)
import importlib.util

_REQUIRED = ("numpy", "pandas", "pacmap", "matplotlib", "plotly", "sklearn")
if any(importlib.util.find_spec(m) is None for m in _REQUIRED):
    %pip install -q numpy pandas pacmap matplotlib plotly scikit-learn

In [2]:
from __future__ import annotations

import ast
import re
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
import pacmap

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams["figure.figsize"] = (10, 7)
plt.rcParams["font.family"] = "AppleGothic" if Path("/System/Library/Fonts").exists() else "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "embedding":
    NOTEBOOK_DIR = Path("./")

ORIGIN_DIR = NOTEBOOK_DIR / "origin_data"
RESULT_DIR = NOTEBOOK_DIR / "result_data_2"

ONTOLOGY_CSV = ORIGIN_DIR / "movie_ontology.csv"

LABEL_TYPES: dict[str, tuple[str, str]] = {
    "genre": ("genres", "label_genre"),
    "theme": ("themes", "label_theme"),  # ontology themes = category 축
    "mood": ("moods", "label_mood"),
}

PACMAP_BASE = dict(MN_ratio=0.5, FP_ratio=2.0, random_state=42)
PACMAP_NEIGHBORS_MIN = 10
PACMAP_NEIGHBORS_MAX = 200
SILHOUETTE_SAMPLE = 2000

In [3]:
def parse_ontology_csv(path: Path) -> pd.DataFrame:
    """리스트 필드에 쉼표가 포함된 ontology CSV를 정규식으로 파싱한다."""
    pattern = re.compile(r'^"([^"]+)", "([^"]*)", (\[.*\]), (\[.*\]), (\[.*\])$')
    rows: list[dict[str, Any]] = []
    bad_lines: list[int] = []

    for lineno, line in enumerate(path.read_text(encoding="utf-8").splitlines()[1:], start=2):
        match = pattern.match(line.strip())
        if not match:
            bad_lines.append(lineno)
            continue
        movie_id, title, genres_raw, themes_raw, moods_raw = match.groups()
        rows.append(
            {
                "movie_id": movie_id.strip(),
                "title": title.strip(),
                "genres": ast.literal_eval(genres_raw),
                "themes": ast.literal_eval(themes_raw),
                "moods": ast.literal_eval(moods_raw),
            }
        )

    if bad_lines:
        raise ValueError(f"ontology 파싱 실패 라인: {bad_lines[:5]} (총 {len(bad_lines)}건)")
    return pd.DataFrame(rows)


def primary_label(values: Any) -> str:
    if values is None or (isinstance(values, float) and np.isnan(values)):
        return "Unknown"
    if isinstance(values, (list, tuple)):
        cleaned = [str(v).strip() for v in values if str(v).strip()]
        return sorted(cleaned)[0] if cleaned else "Unknown"
    return str(values).strip() or "Unknown"


def load_result_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["genres"] = df["genres"].apply(ast.literal_eval)
    return df


def embedding_matrix(df: pd.DataFrame) -> tuple[np.ndarray, list[str]]:
    emb_cols = [c for c in df.columns if c.startswith("embedding_")]
    if not emb_cols:
        raise ValueError("embedding_* 컬럼이 없습니다.")
    emb_cols = sorted(emb_cols, key=lambda c: int(c.split("_")[1]))
    return df[emb_cols].to_numpy(dtype=float), emb_cols


def build_model_frame(result_df: pd.DataFrame, ontology: pd.DataFrame) -> pd.DataFrame:
    merged = result_df.merge(
        ontology[["movie_id", "themes", "moods"]],
        on="movie_id",
        how="left",
    )

    for _, (src_col, label_col) in LABEL_TYPES.items():
        merged[label_col] = merged[src_col].apply(primary_label)

    return merged


def l2_normalize(X: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return X / norms


def pacmap_n_neighbors(n_samples: int, labels: pd.Series) -> int:
    """클래스 수·클래스별 샘플 분포에 맞춰 PaCMAP n_neighbors를 산출한다."""
    counts = labels.value_counts()
    n_classes = len(counts)
    if n_samples < 3 or n_classes < 2:
        return max(2, n_samples - 1)

    median_size = float(counts.median())
    # 클래스당 중위 샘플 수를 기준으로 지역 이웃 규모 결정
    k = int(round(median_size * 2.0))
    return max(PACMAP_NEIGHBORS_MIN, min(PACMAP_NEIGHBORS_MAX, k, n_samples - 1))


def run_pacmap(X: np.ndarray, n_components: int, n_neighbors: int) -> np.ndarray:
    reducer = pacmap.PaCMAP(
        n_components=n_components,
        n_neighbors=n_neighbors,
        **PACMAP_BASE,
    )
    return reducer.fit_transform(l2_normalize(X), init="pca")


def clustering_metrics(X: np.ndarray, labels: pd.Series) -> dict[str, float | int | None]:
    y = pd.factorize(labels.astype(str))[0]
    n_classes = len(np.unique(y))
    n_samples = len(y)

    if n_classes < 2 or n_samples < n_classes + 1:
        return {
            "n_samples": n_samples,
            "n_classes": n_classes,
            "silhouette_cosine": None,
            "calinski_harabasz": None,
            "davies_bouldin": None,
        }

    sample_size = min(SILHOUETTE_SAMPLE, n_samples)
    sil = float(silhouette_score(X, y, metric="cosine", sample_size=sample_size, random_state=42))
    ch = float(calinski_harabasz_score(X, y))
    db = float(davies_bouldin_score(X, y))
    return {
        "n_samples": n_samples,
        "n_classes": n_classes,
        "silhouette_cosine": sil,
        "calinski_harabasz": ch,
        "davies_bouldin": db,
    }

In [4]:
ontology_df = parse_ontology_csv(ONTOLOGY_CSV)
print(f"ontology 영화 수: {len(ontology_df):,}")

result_files = sorted(RESULT_DIR.glob("result-*.csv"))
if not result_files:
    raise FileNotFoundError(f"결과 CSV가 없습니다: {RESULT_DIR}")

model_frames: dict[str, pd.DataFrame] = {}
load_report_rows: list[dict[str, Any]] = []

for path in result_files:
    model_name = path.stem.replace("result-", "")
    raw_df = load_result_csv(path)
    frame = build_model_frame(raw_df, ontology_df)
    X, emb_cols = embedding_matrix(frame)

    model_frames[model_name] = frame
    load_report_rows.append(
        {
            "model": model_name,
            "result_rows": len(raw_df),
            "used_rows": len(frame),
            "embedding_dim": len(emb_cols),
            "ontology_matched": int(frame["themes"].notna().sum()),
        }
    )

load_report_df = pd.DataFrame(load_report_rows)
display(load_report_df)

ontology 영화 수: 8,300


/var/folders/vc/lxv_xsmd72d28lk6hlwg1xzw0000gn/T/ipykernel_70721/1827501261.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged[label_col] = merged[src_col].apply(primary_label)
/var/folders/vc/lxv_xsmd72d28lk6hlwg1xzw0000gn/T/ipykernel_70721/1827501261.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged[label_col] = merged[src_col].apply(primary_label)
/var/folders/vc/lxv_xsmd72d28lk6hlwg1xzw0000gn/T/ipykernel_70721/1827501261.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

,model,result_rows,used_rows,embedding_dim,ontology_matched
0,bge-m3,6204,6204,1024,6204
1,qwen3-0_6b,6204,6204,1024,6204
2,qwen3-4b-1024,6204,6204,1024,6204


In [5]:
def plot_matplotlib_2d(df: pd.DataFrame, label_col: str, title: str) -> None:
    fig, ax = plt.subplots(figsize=(11, 8))
    labels = df[label_col].astype(str)
    for name, group in df.groupby(labels, sort=False):
        ax.scatter(group["pacmap_x"], group["pacmap_y"], s=12, alpha=0.65, label=name)
    ax.set_xlabel("PaCMAP-1")
    ax.set_ylabel("PaCMAP-2")
    ax.set_title(title)
    if labels.nunique() <= 20:
        ax.legend(markerscale=2, fontsize=8, loc="best")
    plt.tight_layout()
    plt.show()


def plot_matplotlib_3d(df: pd.DataFrame, label_col: str, title: str) -> None:
    fig = plt.figure(figsize=(11, 8))
    ax = fig.add_subplot(111, projection="3d")
    labels = df[label_col].astype(str)
    for name, group in df.groupby(labels, sort=False):
        ax.scatter(
            group["pacmap_x"],
            group["pacmap_y"],
            group["pacmap_z"],
            s=10,
            alpha=0.6,
            label=name,
        )
    ax.set_xlabel("PaCMAP-1")
    ax.set_ylabel("PaCMAP-2")
    ax.set_zlabel("PaCMAP-3")
    ax.set_title(title)
    if labels.nunique() <= 15:
        ax.legend(markerscale=2, fontsize=7, loc="best")
    plt.tight_layout()
    plt.show()


def plot_plotly_2d(df: pd.DataFrame, label_col: str, title: str) -> go.Figure:
    fig = px.scatter(
        df,
        x="pacmap_x",
        y="pacmap_y",
        color=label_col,
        hover_data=["title", "movie_id"],
        title=title,
        opacity=0.75,
        height=650,
    )
    fig.update_traces(marker=dict(size=6))
    fig.update_layout(legend_title_text=label_col)
    fig.show()
    return fig


def plot_plotly_3d(df: pd.DataFrame, label_col: str, title: str) -> go.Figure:
    fig = px.scatter_3d(
        df,
        x="pacmap_x",
        y="pacmap_y",
        z="pacmap_z",
        color=label_col,
        hover_data=["title", "movie_id"],
        title=title,
        opacity=0.8,
        height=700,
    )
    fig.update_traces(marker=dict(size=1))
    fig.update_layout(legend_title_text=label_col)
    fig.show()
    return fig

In [6]:
metric_rows: list[dict[str, Any]] = []
pacmap_cache: dict[str, dict[str, dict[str, pd.DataFrame]]] = {}
neighbor_rows: list[dict[str, Any]] = []

for model_name, frame in model_frames.items():
    X, _ = embedding_matrix(frame)
    pacmap_cache[model_name] = {}

    for label_name, (_, label_col) in LABEL_TYPES.items():
        labels = frame[label_col]
        class_counts = labels.value_counts()
        n_neighbors = pacmap_n_neighbors(len(frame), labels)

        coords_2d = run_pacmap(X, n_components=2, n_neighbors=n_neighbors)
        coords_3d = run_pacmap(X, n_components=3, n_neighbors=n_neighbors)

        vis_2d = frame.copy()
        vis_2d[["pacmap_x", "pacmap_y"]] = coords_2d

        vis_3d = frame.copy()
        vis_3d[["pacmap_x", "pacmap_y", "pacmap_z"]] = coords_3d

        pacmap_cache[model_name][label_name] = {"2d": vis_2d, "3d": vis_3d}

        neighbor_rows.append(
            {
                "model": model_name,
                "label_type": label_name,
                "n_samples": len(frame),
                "n_classes": int(labels.nunique()),
                "min_class_size": int(class_counts.min()),
                "median_class_size": float(class_counts.median()),
                "n_neighbors": n_neighbors,
            }
        )

        emb_metrics = clustering_metrics(X, labels)
        metric_rows.append(
            {
                "model": model_name,
                "label_type": label_name,
                "space": "embedding",
                **emb_metrics,
            }
        )

neighbor_df = pd.DataFrame(neighbor_rows)
print("=== 라벨별 PaCMAP n_neighbors (클래스 분포 기반) ===")
display(neighbor_df)

metrics_df = pd.DataFrame(metric_rows)
metrics_pivot = metrics_df.pivot_table(
    index=["model", "label_type"],
    columns="space",
    values="silhouette_cosine",
)

print("=== 모델 × 라벨별 Silhouette (cosine, 높을수록 군집 분리 우수) ===")
display(metrics_pivot.round(4))
display(metrics_df.sort_values(["model", "label_type", "space"]).reset_index(drop=True))

Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.


=== 라벨별 PaCMAP n_neighbors (클래스 분포 기반) ===


,model,label_type,n_samples,n_classes,min_class_size,median_class_size,n_neighbors
0,bge-m3,genre,6204,44,1,24.5,49
1,bge-m3,theme,6204,50,1,27.5,55
2,bge-m3,mood,6204,33,1,58.0,116
3,qwen3-0_6b,genre,6204,44,1,24.5,49
4,qwen3-0_6b,theme,6204,50,1,27.5,55
5,qwen3-0_6b,mood,6204,33,1,58.0,116
6,qwen3-4b-1024,genre,6204,44,1,24.5,49
7,qwen3-4b-1024,theme,6204,50,1,27.5,55
8,qwen3-4b-1024,mood,6204,33,1,58.0,116


=== 모델 × 라벨별 Silhouette (cosine, 높을수록 군집 분리 우수) ===


space                     embedding
model         label_type           
bge-m3        genre         -0.0649
              mood          -0.1015
              theme         -0.1313
qwen3-0_6b    genre         -0.0638
              mood          -0.1182
              theme         -0.1572
qwen3-4b-1024 genre         -0.0716
              mood          -0.0867
              theme         -0.1453

,model,label_type,space,n_samples,n_classes,silhouette_cosine,calinski_harabasz,davies_bouldin
0,bge-m3,genre,embedding,6204,44,-0.064851,8.609724,5.607118
1,bge-m3,mood,embedding,6204,33,-0.101549,6.852159,8.589141
2,bge-m3,theme,embedding,6204,50,-0.131335,4.168286,7.463744
3,qwen3-0_6b,genre,embedding,6204,44,-0.063826,29.636133,3.269001
4,qwen3-0_6b,mood,embedding,6204,33,-0.118179,15.856702,6.947823
5,qwen3-0_6b,theme,embedding,6204,50,-0.157238,7.736219,6.378165
6,qwen3-4b-1024,genre,embedding,6204,44,-0.071624,24.768182,3.643305
7,qwen3-4b-1024,mood,embedding,6204,33,-0.086706,14.987836,6.647485
8,qwen3-4b-1024,theme,embedding,6204,50,-0.145314,7.939575,6.022963


## 리포트 해석 가이드

- **Silhouette (cosine)**: -1~1, 1에 가까울수록 동일 라벨끼리 응집·이질 라벨과 분리가 잘 됨. (원본 임베딩 공간 기준)
- **Calinski-Harabasz**: 값이 클수록 군집 간 분산 대비 군집 내 분산이 작음.
- **Davies-Bouldin**: 값이 작을수록 군집 품질이 좋음.
- **PaCMAP**: 전역·지역 구조를 함께 보존하는 차원 축소. `n_neighbors`는 라벨별 클래스 수·중위 클래스 크기에 따라 자동 산출합니다 (`median_class_size × 2`, 10~200 범위).
- 라벨은 배열의 **대표값(primary, 사전순 첫 항목)** 으로 색상을 지정했습니다.
- `theme` 축은 ontology의 `themes` 필드이며, 요청하신 category에 해당합니다.

In [7]:
for model_name in sorted(pacmap_cache):
    print("\n" + "=" * 80)
    print(f"MODEL: {model_name}")
    print("=" * 80)

    for label_name, (_, label_col) in LABEL_TYPES.items():
        vis_2d = pacmap_cache[model_name][label_name]["2d"]
        vis_3d = pacmap_cache[model_name][label_name]["3d"]
        n_neighbors = neighbor_df.loc[
            (neighbor_df["model"] == model_name) & (neighbor_df["label_type"] == label_name),
            "n_neighbors",
        ].iloc[0]

        title_base = f"{model_name} | {label_name} (n_neighbors={n_neighbors})"

        # print(f"\n--- {label_name.upper()} | Matplotlib 2D ---")
        # plot_matplotlib_2d(vis_2d, label_col, f"{title_base} (PaCMAP 2D, matplotlib)")

        # print(f"--- {label_name.upper()} | Matplotlib 3D ---")
        # plot_matplotlib_3d(vis_3d, label_col, f"{title_base} (PaCMAP 3D, matplotlib)")

        print(f"--- {label_name.upper()} | Plotly 2D ---")
        plot_plotly_2d(vis_2d, label_col, f"{title_base} (PaCMAP 2D, plotly)")

        # print(f"--- {label_name.upper()} | Plotly 3D ---")
        # plot_plotly_3d(vis_3d, label_col, f"{title_base} (PaCMAP 3D, plotly)")


MODEL: bge-m3
--- GENRE | Plotly 2D ---


--- THEME | Plotly 2D ---


--- MOOD | Plotly 2D ---



MODEL: qwen3-0_6b
--- GENRE | Plotly 2D ---


--- THEME | Plotly 2D ---


--- MOOD | Plotly 2D ---



MODEL: qwen3-4b-1024
--- GENRE | Plotly 2D ---


--- THEME | Plotly 2D ---


--- MOOD | Plotly 2D ---
